In [1]:
import os
import pandas as pd
import yfinance as yf
from ollama import Client
from dotenv import load_dotenv

In [2]:
ticker = "VCB"

In [3]:
yf_ticker = f"{ticker}.VN"

In [4]:
stock = yf.Ticker(yf_ticker)

In [5]:
info = stock.info

In [6]:
info

{'address1': 'Vietcombank Tower',
 'address2': '19th Floor 198 Tran Quang Khai street Hoan Kiem District',
 'city': 'Hanoi',
 'country': 'Vietnam',
 'phone': '84 24 3934 3137',
 'website': 'https://www.vietcombank.com.vn',
 'industry': 'Banks - Regional',
 'industryKey': 'banks-regional',
 'industryDisp': 'Banks - Regional',
 'sector': 'Financial Services',
 'sectorKey': 'financial-services',
 'sectorDisp': 'Financial Services',
 'longBusinessSummary': 'Joint Stock Commercial Bank for Foreign Trade of Vietnam, together with its subsidiaries, provides banking products and services in Vietnam. The company accepts short, medium, and long-term deposits from organizations and individuals; trade of gold bars; and engages in financial and office leasing, and money remittance activities. It also provides account services; digital banking services; payment, corporate, and credit cards; consumer, home, auto, and business loans; and investment and savings insurance. In addition, the company offer

In [7]:
fa_data = {
    "Chỉ số": [
        "P/E Hiện tại (Trailing PE)", 
        "P/E Dự phóng (Forward PE)",
        "P/B (Giá/Sổ sách)", 
        "ROE (Lợi nhuận/Vốn chủ) %", 
        "ROA (Lợi nhuận/Tài sản) %", 
        "Biên lợi nhuận ròng %",
        "Tăng trưởng Doanh thu %",
        "Tăng trưởng Lợi nhuận Quý %",
        "Tỷ suất Cổ tức %"
    ],
    "Giá trị Hiện tại": [
        round(info.get("trailingPE", 0), 2) if info.get("trailingPE") else "N/A",
        round(info.get("forwardPE", 0), 2) if info.get("forwardPE") else "N/A",
        round(info.get("priceToBook", 0), 2) if info.get("priceToBook") else "N/A",
        round(info.get("returnOnEquity", 0) * 100, 2) if info.get("returnOnEquity") else "N/A",
        round(info.get("returnOnAssets", 0) * 100, 2) if info.get("returnOnAssets") else "N/A",
        round(info.get("profitMargins", 0) * 100, 2) if info.get("profitMargins") else "N/A",
        round(info.get("revenueGrowth", 0) * 100, 2) if info.get("revenueGrowth") else "N/A",
        round(info.get("earningsQuarterlyGrowth", 0) * 100, 2) if info.get("earningsQuarterlyGrowth") else "N/A",
        round(info.get("dividendYield", 0) * 100, 2) if info.get("dividendYield") else "N/A"
    ]
}

In [8]:
fa_data

{'Chỉ số': ['P/E Hiện tại (Trailing PE)',
  'P/E Dự phóng (Forward PE)',
  'P/B (Giá/Sổ sách)',
  'ROE (Lợi nhuận/Vốn chủ) %',
  'ROA (Lợi nhuận/Tài sản) %',
  'Biên lợi nhuận ròng %',
  'Tăng trưởng Doanh thu %',
  'Tăng trưởng Lợi nhuận Quý %',
  'Tỷ suất Cổ tức %'],
 'Giá trị Hiện tại': [15.43, 8.57, 2.21, 16.73, 1.55, 50.79, 3.4, 0.7, 76.0]}

In [9]:
df = pd.DataFrame(fa_data)
df

,Chỉ số,Giá trị Hiện tại
0,P/E Hiện tại (Trailing PE),15.43
1,P/E Dự phóng (Forward PE),8.57
2,P/B (Giá/Sổ sách),2.21
3,ROE (Lợi nhuận/Vốn chủ) %,16.73
4,ROA (Lợi nhuận/Tài sản) %,1.55
5,Biên lợi nhuận ròng %,50.79
6,Tăng trưởng Doanh thu %,3.40
7,Tăng trưởng Lợi nhuận Quý %,0.70
8,Tỷ suất Cổ tức %,76.00


In [10]:
markdown_table = df.to_markdown(index=False)
print(markdown_table)

| Chỉ số                      |   Giá trị Hiện tại |
|:----------------------------|-------------------:|
| P/E Hiện tại (Trailing PE)  |              15.43 |
| P/E Dự phóng (Forward PE)   |               8.57 |
| P/B (Giá/Sổ sách)           |               2.21 |
| ROE (Lợi nhuận/Vốn chủ) %   |              16.73 |
| ROA (Lợi nhuận/Tài sản) %   |               1.55 |
| Biên lợi nhuận ròng %       |              50.79 |
| Tăng trưởng Doanh thu %     |               3.4  |
| Tăng trưởng Lợi nhuận Quý % |               0.7  |
| Tỷ suất Cổ tức %            |              76    |


In [11]:
SYSTEM_PROMPT = """
Bạn là một Chuyên gia Phân tích Cơ bản (Fundamental Analyst) tài năng tại thị trường chứng khoán Việt Nam.
Nhiệm vụ của bạn là đọc các chỉ số tài chính của doanh nghiệp và đưa ra BÁO CÁO ĐÁNH GIÁ SỨC KHỎE TÀI CHÍNH & ĐỊNH GIÁ.

Mục tiêu phân tích:
1. Định giá (Valuation): Đánh giá P/E (Hiện tại & Dự phóng) và P/B xem cổ phiếu đang đắt hay rẻ.
2. Hiệu quả sinh lời (Profitability): Phân tích ROE, ROA, Biên lợi nhuận.
3. Tăng trưởng (Growth): Đánh giá đà tăng trưởng doanh thu và lợi nhuận.
4. Cổ tức & Rủi ro: Đánh giá tỷ suất cổ tức.

Ràng buộc nghiêm ngặt:
- Đi thẳng vào vấn đề, không tốn thời gian giải thích định nghĩa các chỉ số (VD: Không cần giải thích P/E là gì).
- Bắt buộc phải có phần "Kết luận & Khuyến nghị Đầu tư" (MUA TÍCH LŨY / THEO DÕI / KHÔNG NÊN ĐẦU TƯ).
- Tuyệt đối chỉ dựa vào số liệu trong bảng, không bịa đặt hay suy diễn số liệu từ bên ngoài.
"""

In [12]:
user_prompt = f"""
Thực hiện đánh giá cơ bản cho mã cổ phiếu: {ticker}
Dưới đây là các chỉ số tài chính quan trọng hiện tại:

{markdown_table}

Hãy phân tích và trình bày báo cáo theo đúng cấu trúc Markdown sau:

### I. Đánh giá Định giá (Valuation)
- (Phân tích P/E và P/B)

### II. Hiệu quả Sinh lời & Tăng trưởng
- (Phân tích ROE, ROA, Biên lợi nhuận)
- (Phân tích tốc độ tăng trưởng)

### III. Sức khỏe Tài chính & Cổ tức
- (Đánh giá mức độ an toàn và tỷ suất cổ tức)

### IV. Kết luận & Khuyến nghị
- **Trạng thái:** (Tuyệt vời / Tốt / Trung bình / Rủi ro)
- **Hành động:** (MUA TÍCH LŨY / THEO DÕI / KHÔNG NÊN ĐẦU TƯ - Kèm giải thích ngắn gọn trong 1 câu)
"""

In [13]:
messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': user_prompt},
]

In [14]:
load_dotenv()

True

In [15]:
HOST = os.getenv("OLLAMA_HOST")
API_KEY = os.getenv("OLLAMA_API_KEY")

In [16]:
client = Client(
    host=HOST,
    headers={'Authorization': f'Bearer {API_KEY}'}
)

In [17]:
MODEL = os.getenv("OLLAMA_MODEL")

In [18]:
if MODEL is None:
    raise ValueError("OLLAMA_MODEL environment variable is not set")

response = client.chat(
    model=MODEL,
    messages=messages,
)

In [19]:
print(response['message']['content'])

### I. Đánh giá Định giá (Valuation)
- **P/E:** P/E hiện tại ở mức 15.43, nhưng P/E dự phóng giảm mạnh xuống còn 8.57. Điều này cho thấy kỳ vọng lợi nhuận trong tương lai sẽ tăng trưởng mạnh, khiến mức giá hiện tại trở nên hấp dẫn hơn.
- **P/B:** Chỉ số 2.21 cho thấy thị trường đang định giá cổ phiếu cao hơn 2 lần giá trị sổ sách, phản ánh uy tín và vị thế đầu ngành của VCB.

### II. Hiệu quả Sinh lời & Tăng trưởng
- **Hiệu quả sinh lời:** 
    - ROE đạt 16.73% và ROA đạt 1.55%, cho thấy khả năng quản trị vốn và tài sản hiệu quả. 
    - Biên lợi nhuận ròng cực cao (50.79%), khẳng định khả năng tối ưu hóa chi phí và kiểm soát rủi ro vận hành xuất sắc.
- **Tăng trưởng:** Đà tăng trưởng đang ở mức thấp và chậm lại. Tăng trưởng doanh thu (3.4%) và lợi nhuận quý (0.7%) cho thấy doanh nghiệp đang trong giai đoạn đi ngang hoặc bão hòa ngắn hạn.

### III. Sức khỏe Tài chính & Cổ tức
- **Sức khỏe tài chính:** Các chỉ số sinh lời ổn định và biên lợi nhuận dày cho thấy sức khỏe tài chính rất vững